In [16]:
import zarr
import numpy as np

In [21]:
import os
HARPS_DATA = '/g/data/y89/mj8805/runs/bigharps/outputs/shards'

In [23]:
chunks = [o for o in os.listdir(HARPS_DATA) if o.endswith('.zarr')]

In [24]:
o = zarr.open_group(os.path.join(HARPS_DATA, chunks[0]))
o.tree()

/
├── a (12500,) StringDType()
├── base_name (12500,) StringDType()
├── c (12500,) StringDType()
├── calculation_mode (12500,) StringDType()
├── continuum (12500, 310001) float32
├── feh (12500,) float64
├── flux (12500, 310001) float32
├── global_index (12500,) int64
├── grid_version (12500,) StringDType()
├── lam_max (12500,) float64
├── lam_min (12500,) float64
├── lam_step (12500,) float64
├── logg (12500,) float64
├── message (12500,) StringDType()
├── mode (12500,) StringDType()
├── mu_selected (12500,) float32
├── mu_selected_index (12500,) int16
├── n (12500,) StringDType()
├── o (12500,) StringDType()
├── output_mode (12500,) StringDType()
├── provenance
│   ├── atmosphere_manifest.json () StringDType()
│   ├── canonical_config.yaml () StringDType()
│   ├── environment.txt () StringDType()
│   ├── linelist_manifest.json () StringDType()
│   ├── software_manifest.json () StringDType()
│   └── synthesis_config.yaml () StringDType()
├── r (12500,) StringDType()
├── s (12500,) StringDType()
├── status (12500,) StringDType()
├── t_value (12500,) StringDType()
├── teff (12500,) int64
├── turbvel (12500,) StringDType()
└── wavelength (310001,) float64

In [33]:
import os
import numpy as np
import zarr
import numcodecs as zc

# ---------------------------
# CONFIG
# ---------------------------

MERGED_NAME = "harps_merged_2.zarr"
ROW_CHUNK = 128   # tune: 64–256 depending on memory

merged_path = os.path.join(HARPS_DATA, MERGED_NAME)

# ---------------------------
# HELPERS
# ---------------------------
def write_string_array(group, name, data):
    arr = np.asarray(data)

    # convert to bytes (UTF-8)
    if arr.dtype.kind in {'U', 'O'}:
        arr = np.asarray([str(x).encode('utf-8') for x in arr])

    group.create_array(name, data=arr)

def write_scalar_string(group, name, value):
    if isinstance(value, bytes):
        b = value
    else:
        b = str(value).encode('utf-8')

    arr = np.frombuffer(b, dtype=np.uint8)

    group.create_array(name, data=arr)

def create_numeric_array(group, name, shape, dtype, chunks):
    return group.create_array(
        name,
        shape=shape,
        dtype=dtype,
        chunks=chunks
    )

# ---------------------------
# OPEN INPUT GROUPS
# ---------------------------
groups = [zarr.open_group(os.path.join(HARPS_DATA, c), mode='r') for c in chunks]

total_spectra = sum(g['flux'].shape[0] for g in groups)
n_wavelength = groups[0]['wavelength'].shape[0]

# Count numeric parameters (excluding string columns like a, c, n, o, r, s)
numeric_param_names = [name for name in groups[0].keys() 
                       if name in ['teff', 'logg', 'feh', 'vmicro'] or 
                       (name not in ['a', 'c', 'n', 'o', 'r', 's', 'flux', 'continuum', 
                                     'wavelength', 'global_index', 'model_id', 'mu_selected', 
                                     'mu_selected_index', 'base_name', 'calculation_mode', 
                                     'grid_version', 'lam_max', 'lam_min', 'lam_step', 
                                     'message', 'mode', 'output_mode', 'status', 't_value', 
                                     'turbvel', 'provenance'])]

n_params = len(numeric_param_names)

print(f"Total spectra: {total_spectra}")
print(f"Wavelength points: {n_wavelength}")
print(f"Parameters: {n_params}")

# ---------------------------
# CREATE OUTPUT GROUP
# ---------------------------
merged = zarr.open_group(merged_path, mode='w')

# ---------------------------
# COPY SHARED ARRAYS
# ---------------------------
merged['wavelength'] = groups[0]['wavelength'][:]

# Create param_names from numeric parameters
merged.create_array('param_names', data=np.array(numeric_param_names, dtype='U32'))

# ---------------------------
# METADATA (SAFE STRINGS)
# ---------------------------
meta = merged.create_group('metadata')

# Handle metadata - check if it exists in source
if 'metadata' in groups[0]:
    if 'physics_hash' in groups[0]['metadata']:
        write_scalar_string(meta, 'physics_hash', groups[0]['metadata']['physics_hash'][()])
    if 'schema_version' in groups[0]['metadata']:
        write_scalar_string(meta, 'schema_version', groups[0]['metadata']['schema_version'][()])

# ---------------------------
# PROVENANCE
# ---------------------------
prov_group = merged.create_group('provenance')

if 'provenance' in groups[0]:
    for key in groups[0]['provenance'].keys():
        val = groups[0]['provenance'][key][()]
        
        if isinstance(val, (str, bytes)):
            write_scalar_string(prov_group, key, val)
        else:
            prov_group.create_array(key, data=np.asarray(val))

# ---------------------------
# CREATE DATA ARRAYS
# ---------------------------
create_numeric_array(
    merged, 'flux',
    shape=(total_spectra, n_wavelength),
    dtype=groups[0]['flux'].dtype,
    chunks=(ROW_CHUNK, n_wavelength)
)

create_numeric_array(
    merged, 'continuum',
    shape=(total_spectra, n_wavelength),
    dtype=groups[0]['continuum'].dtype,
    chunks=(ROW_CHUNK, n_wavelength)
)

create_numeric_array(
    merged, 'params',
    shape=(total_spectra, n_params),
    dtype=np.float32,
    chunks=(ROW_CHUNK, n_params)
)

for name in ['global_index', 'model_id', 'mu_selected', 'mu_selected_index']:
    if name in groups[0]:
        create_numeric_array(
            merged, name,
            shape=(total_spectra,),
            dtype=groups[0][name].dtype,
            chunks=(1024,)
        )

# ---------------------------
# PARAMETER COLUMNS
# ---------------------------
param_cols = merged.create_group('parameter_columns')

# Get all parameter column names from first group
all_param_names = ['teff', 'logg', 'feh', 'vmicro', 'a', 'c', 'n', 'o', 'r', 's']

for param in all_param_names:
    if param in groups[0]:
        # Determine dtype - string columns need special handling
        create_numeric_array(
            param_cols, param,
            shape=(total_spectra,),
            dtype=np.float32,
            chunks=(1024,)
        )

# ---------------------------
# STREAMING MERGE
# ---------------------------
offset = 0

for i, g in enumerate(groups):
    n_spec = g['flux'].shape[0]
    print(f"Processing chunk {i+1}/{len(groups)} ({n_spec} spectra)")

    for start in range(0, n_spec, ROW_CHUNK):
        end = min(start + ROW_CHUNK, n_spec)

        in_sl = slice(start, end)
        out_sl = slice(offset + start, offset + end)

        # 2D arrays
        merged['flux'][out_sl] = g['flux'][in_sl]
        merged['continuum'][out_sl] = g['continuum'][in_sl]
        
        # Build params array from numeric columns
        params_data = []
        for param_name in numeric_param_names:
            if param_name in g:
                params_data.append(g[param_name][in_sl])
        
        if params_data:
            merged['params'][out_sl] = np.column_stack(params_data)

        # 1D arrays
        for name in ['global_index', 'model_id', 'mu_selected', 'mu_selected_index']:
            if name in g and name in merged:
                merged[name][out_sl] = g[name][in_sl]

        # parameter columns
        for param in all_param_names:
            if param in g and param in param_cols:
                data = g[param][in_sl]

                # robust conversion (handles bytes or str)
                data = np.array([
                    float(x.decode('utf-8') if isinstance(x, bytes) else x)
                    for x in data
                ], dtype=np.float32)

                param_cols[param][out_sl] = data

    offset += n_spec

# ---------------------------
# VALIDATION
# ---------------------------
print("Running validation...")

o = zarr.open_group(merged_path, mode='r')

print("✅ Merge complete and validated!")
print(o.tree())

Total spectra: 87500
Wavelength points: 310001
Parameters: 3
Processing chunk 1/7 (12500 spectra)
Processing chunk 2/7 (12500 spectra)
Processing chunk 3/7 (12500 spectra)
Processing chunk 4/7 (12500 spectra)
Processing chunk 5/7 (12500 spectra)
Processing chunk 6/7 (12500 spectra)


KeyboardInterrupt: 

In [11]:
o = zarr.open_group(merged_path, mode='r')

In [12]:
o.tree()

/
├── continuum (56541, 310001) float32
├── flux (56541, 310001) float32
├── global_index (56541,) int64
├── metadata
│   ├── physics_hash (64,) uint8
│   └── schema_version (5,) uint8
├── model_id (56541,) uint64
├── mu_selected (56541,) float32
├── mu_selected_index (56541,) int16
├── param_names (10,) <U32
├── parameter_columns
│   ├── a (56541,) float32
│   ├── c (56541,) float32
│   ├── feh (56541,) float32
│   ├── logg (56541,) float32
│   ├── n (56541,) float32
│   ├── o (56541,) float32
│   ├── r (56541,) float32
│   ├── s (56541,) float32
│   ├── teff (56541,) float32
│   └── vmicro (56541,) float32
├── params (56541, 10) float32
├── provenance
│   ├── atmosphere_manifest.json (251,) uint8
│   ├── canonical_config.yaml (995,) uint8
│   ├── environment.txt (169,) uint8
│   ├── linelist_manifest.json (5773,) uint8
│   ├── software_manifest.json (321,) uint8
│   └── synthesis_config.yaml (1722,) uint8
└── wavelength (310001,) float32

In [15]:
o['mu_selected'][0]

array(0.974726, dtype=float32)

In [37]:
o['mu_selected'][:10]

array([0.974726, 0.468138, 0.974726, 0.916958, 0.830825, 0.830825,
       0.468138, 1.      , 0.830825, 0.468138], dtype=float32)

In [38]:
o['continuum'][0], o['flux'][0], o['flux'][0]/o['continuum'][0]

(array([2.9375430e+06, 2.9375505e+06, 2.9375698e+06, ..., 1.8159909e+06,
        1.8159871e+06, 1.8159798e+06], shape=(310001,), dtype=float32),
 array([2.13107e+06, 2.14732e+06, 2.16164e+06, ..., 1.81590e+06,
        1.81586e+06, 1.81578e+06], shape=(310001,), dtype=float32),
 array([0.72546   , 0.73099   , 0.73586   , ..., 0.99994993, 0.99993   ,
        0.99989   ], shape=(310001,), dtype=float32))